# Survey Tasks and Actual Attributions - 2022

This notebook is doing a similar task like notebook `3_connect_survey_tasks_with_attributions.ipynb`, but now for 2022. We do it in a seperate notebook just for easier analysis. Also, in that 2023 matching notebook, the first step was to match attributions and roster names. Here, the attributions are extracted with an LLM by using the roster names so that step is not needed.

First, matching between surveyed names and roster names is done. Survey participants' full names were hashed. The goal is to connect these hashes with the roster names, so we can match the LLM JSON outputs with them. 

In 2022, there are 3 different attributions structures that teams had on their websites. There are textual, image, and tabular attributions. For the first kind, an LLM was used to extract corresponding Task descriptions, but the latter two had to be manually annotated. Therefore, in the second part of this notebook, text teams JSON (outputs from LLM extraction) was merged with the manually annotated JSON files: 7 validation teams, 12 evaluation teams (because 2 of them had to be manually annotated as well), image teams, and table teams. This merged dataset does not include all teams that competed in 2022, but only the surveyed ones.

Then, BERT and LLM classification was done for all teams. The final dataset, with all teams, members, and their task categories, can be analyzed in a separate notebook, so its distributions are compared to the other years. This dataset was saved at `../results/attributions/attributions_for_surveyed_teams_bert_2022.tsv` for BERT outputs (with probabilities for each task) and `../results/attributions/attributions_llm_2022.tsv` for LLM outputs

Lastly, the dataset containing surveyed teams and members' tasks is merged with survey responses. A binary `TaskPerformed` column will have info whether a survey respondant has performed a task or not. The final combined attributions and survey dataset is saved at `../results/survey_tasks_and_attributions/2022_survey_tasks_and_attributions_combined_bert.tsv` (or _llm).

In [263]:
import ast
import pandas as pd
import hashlib
from rapidfuzz import process, fuzz
import unicodedata
import json
import math
import unicodedata

## 1. Matching Survey and Roster Names

The goal is to match `survey data` which only has user IDs with `roster data` which has full names of members. The matching is done by the helper table `user_info` (non anonymized version) that contains both the IDs and full names. The issue is that not all names in `user_info` and `roster data` are completely the same: some are spelled differently and some were changed completely after the 2022 survey (since the roster data was collected in 2025).

We will first explore the 3 datasets. We will keep only the `user_info` for people present in the survey, and we will keep the rosters only for teams present in the survey (just not to work with the entire unnecessary data). 

Then, we will continue with the merging + fuzzy name matching between `user_info` and `roster data`. Finally, this will give us a dataframe with full names and user IDs from `user_info`, and full names from `roster data`. We will call this dataframe `participant_info` as it will have info only on survey participants. We will anonymize the fullname columns in this dataframe, and save it on `../data/igem_ties_surveys/2022/participant_info.tsv`. 

By merging on the user ID, our survey data can now have an anonymized roster full name column. This can later be merged with the BERT task outputs, where full names will be anonymized using the same algorithm. Now, the merged dataset will have the binary `TaskPerformed` column.

### 1.1 Data

In [266]:
df_roster_2022 = pd.read_table("../data/attributions/2022_attributions/team_rosters_2022.tsv")

In [267]:
survey_tasks_df = pd.read_csv("../data/igem_ties_surveys/2022/survey_tasks.csv")
survey_tasks_df

,team,user_id,Task,Certainty,Experience,Preference
0,AFCM-Egypt,5023,Analysis,Completely sure,Between one and two years,Somewhat agree
1,AFCM-Egypt,5023,Background Research,Completely sure,Between three and four years,Somewhat agree
2,AFCM-Egypt,5023,Conceptualization,Somewhat sure,Between one and two years,Somewhat agree
3,AFCM-Egypt,5023,Data Curation,Somewhat sure,Between two and three years,Somewhat agree
4,AFCM-Egypt,5023,Entrepreneurship,Neutral,Less than one year,Somewhat disagree
...,...,...,...,...,...,...
11605,William_and_Mary,9429,Public Engagement,Somewhat sure,Between two and three years,Neutral
11606,William_and_Mary,9429,Safety,Neutral,Less than one year,Somewhat disagree
11607,William_and_Mary,9429,Software,Completely unsure,Less than one year,Neutral
11608,William_and_Mary,9429,Visualization,Completely sure,Between two and three years,Somewhat agree


In [268]:
user_info = pd.read_csv("../data/igem_ties_surveys/2022/user_info_2.csv")

The survey data has a "user_id" column that has a full participant name match in the `user_info.csv` file. So, we will first match roster and user_info names, and then match them to the survey using survey_id. User info has all team members in 2022, but we are only interested in those who have filled out the survey (present in survey tasks data).

In [269]:
# Get surveyed teams
survey_teams_unique = survey_tasks_df[['team']].drop_duplicates().reset_index(drop=True)
surveyed_teams_list = survey_teams_unique['team'].tolist()

# Get rosters of only surveyed teams
df_roster_2022_surveyed = df_roster_2022[df_roster_2022["Team"].isin(surveyed_teams_list)]
df_roster_2022_surveyed_unique = df_roster_2022_surveyed[['Team']].drop_duplicates().reset_index(drop=True)
roster_list_surveyed = df_roster_2022_surveyed_unique['Team'].tolist()

# Check if there are teams that exist in the survey but not in the rosters
set(surveyed_teams_list) - set(roster_list_surveyed)

set()

In [270]:
# Check if there are teams that exist in the rosters but not in user_info
user_info_teams = user_info[['Team']].drop_duplicates().reset_index(drop=True)
user_info_teams_list = user_info_teams['Team'].tolist()

set(roster_list_surveyed) - set(user_info_teams_list)

set()

All teams from survey data and roster data match. 'Interaction-Data-Lab' is just a test survey team that we will drop in the next cell.

In [271]:
# Drop rows where the team name is "Interaction-Data-Lab" and save it to csv

survey_tasks_df = survey_tasks_df[survey_tasks_df['team'] != 'Interaction-Data-Lab']
survey_tasks_df.to_csv('../data/igem_ties_surveys/2022/survey_tasks.csv', index=False)
survey_tasks_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11610 entries, 0 to 11609
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   team        11610 non-null  object
 1   user_id     11610 non-null  int64 
 2   Task        11610 non-null  object
 3   Certainty   11489 non-null  object
 4   Experience  11007 non-null  object
 5   Preference  10779 non-null  object
dtypes: int64(1), object(5)
memory usage: 544.3+ KB


In [272]:
print('Number of surveyed teams in 2022:', len(surveyed_teams_list))
print('Number of surveyed members in 2022:', survey_tasks_df["user_id"].nunique())

Number of surveyed teams in 2022: 112
Number of surveyed members in 2022: 776


In [273]:
df_roster_2022_surveyed

,FullName,TeamID,Team,Year,Role,Username,RosterUUID,UserUUID,UserID
0,Moetaz Sherif Mohamed Radwan Metawea,4140,AFCM-Egypt,2022,Advisor,moetazsherif,226eddff-9b45-481f-b68e-01c1689ab757,65acc482-eb58-4b58-bd85-0b9a6e222586,57587
1,Omar Ahmed Abdalla,4140,AFCM-Egypt,2022,Advisor,meero99999,ef8aec10-3e57-4bff-b3e2-42ee125d165d,dde71e87-6a89-4994-b45f-f698d18f4dc6,62513
2,Ahmad Mahmoud Galal,4140,AFCM-Egypt,2022,Instructor,a7madgalal,060dff49-f053-468c-9c86-f5fb3de4c475,5a2fb348-f03e-4aee-b7e1-3142854087d4,49360
3,Ahmed Elshazly,4140,AFCM-Egypt,2022,Instructor,ahmedelshazly,4c3c590d-c31c-4b95-8116-bd8e7e401456,a72af0ca-62c0-49d1-a42c-8b639bcad712,73408
4,Mahmoud Mohammed AbdelGawad,4140,AFCM-Egypt,2022,Instructor,abdelgawad,722c4469-3270-4ef7-8340-94b4e3bdaf27,ebf4dda2-dd7d-403f-a6ce-bcbd4e021d66,34828
...,...,...,...,...,...,...,...,...,...
7306,Krithika Layagala,4174,William_and_Mary,2022,Student Member,klayagala,138ce050-ee40-4740-9524-afe476f3a79f,815698ea-c69e-4060-96d2-76218cbe6c73,67491
7307,Lin Fang,4174,William_and_Mary,2022,Student Member,linfang,20f7475e-0d64-4da1-a310-eba8abb3ea59,a619c437-9430-4748-9dcb-f66a55504e85,67151
7308,Megan Fleeharty,4174,William_and_Mary,2022,Student Member,msfleeharty,3982be0f-d50f-45f4-848d-4f3a93644d6c,dff0960b-08f2-4d81-b126-5fa15473b75d,67149
7309,Walker Knapp,4174,William_and_Mary,2022,Student Member,walkerknapp,f572bb9a-7bb7-4a25-81c5-fa3b4977d9ee,b10fabf5-76e6-411f-afb7-279d8c3ca0dd,67169


**Note**: The UserID in the roster is not the same as the one used in the survey, so we will rename it to UserIDRoster for clarity. The matching has to be done by full name with user_info first.

In [274]:
# Rename columnn

df_roster_2022_surveyed = df_roster_2022_surveyed.rename(columns={'UserID': 'UserIDRoster'})

In [275]:
# Get only surveyed user_info - user ids exist in survey tasks data

survey_ids_unique = survey_tasks_df[['user_id']].drop_duplicates().reset_index(drop=True)
surveyed_ids_list = survey_ids_unique['user_id'].tolist()

user_info_surveyed = user_info[user_info["UserId"].isin(surveyed_ids_list)]
user_info_surveyed = user_info_surveyed.rename(columns={"FullName": "SurveyName"})
user_info_surveyed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 776 entries, 24 to 7690
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Team        776 non-null    object
 1   UserId      776 non-null    int64 
 2   SurveyName  776 non-null    object
dtypes: int64(1), object(2)
memory usage: 24.2+ KB


In [276]:
df_roster_2022_surveyed = df_roster_2022_surveyed.rename(columns={"FullName": "RosterName"})
df_roster_2022_surveyed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2272 entries, 0 to 7310
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   RosterName    2272 non-null   object
 1   TeamID        2272 non-null   int64 
 2   Team          2272 non-null   object
 3   Year          2272 non-null   int64 
 4   Role          2272 non-null   object
 5   Username      2272 non-null   object
 6   RosterUUID    2272 non-null   object
 7   UserUUID      2272 non-null   object
 8   UserIDRoster  2272 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 177.5+ KB


### 1.2 Merge and Fuzzy Match

In [277]:
# Just to check how many names from the user_info_surveyed do not have a match with a name from the roster data
# This will be done again within the following function that has been modified from the third notebook

merged_check_df = (
    user_info_surveyed
    .merge(
        df_roster_2022_surveyed[["Team", "RosterName", "RosterUUID"]],
        how="left",
        left_on=["Team", "SurveyName"],
        right_on=["Team", "RosterName"],
    )
)

merged_check_df.info()
# merged_check_df[merged_check_df["RosterName"].isna()]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 776 entries, 0 to 775
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Team        776 non-null    object
 1   UserId      776 non-null    int64 
 2   SurveyName  776 non-null    object
 3   RosterName  707 non-null    object
 4   RosterUUID  707 non-null    object
dtypes: int64(1), object(4)
memory usage: 30.4+ KB


There are 71 names in the surveyed version of user_info that do not have an absolute match in the roster data. We will take into account spelling variations of the names between the two dtasets by fuzzy matching the names. By running the following function on `user_info_surveyed` and `df_roster_2022_surveyed`, the two datasets will be first left merged, and then the leftover null names will be fuzzy matched between them.

In [278]:
# Helper function to normalize name 

def normalize_name(s):
    return (
        unicodedata.normalize("NFKD", s)
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )


def merge_and_fuzzy_match_names(df1, column1, df2, column2, team_column="Team", threshold=80):
    """
    Firstly, left merge two datasets df1 and df2: e.g. in our case `user_info_surveyed` and `df_roster_2022_surveyed`, on (Team, FullName).
    After the merging, a fullname column (column2 - e.g. in our case RosterName) will have some null values due to unmatched names.
    
    Fuzzy matches lowercase names from df1[column1] (e.g. user_info_surveyed["SurveyName"]) with df2[column2] (df_roster_2022_surveyed["RosterName"]), scoped by team.
    Adds a new column column2 to df1 with the best name match from df2[column2].
    
    If no match is found above the given threshold, column2 is left empty in the merged df.
    
    Parameters:
        df1 (dataframe): First dataframe with names to match.
        column1 (str): Column name in df1 with names to match.
        df2 (dataframe): Second dataframe with reference names, the name map df.
        column2 (str): Column name in df2 with names to match against.
        team_column: The column containing team names, both in df1 and df2. This column needs to be named the same in both dfs (e.g. "Team")
        threshold: Fuzzy matching threshold, adjust it to 80-90.
    
    Returns:
        merged_df (DataFrame): Original df1 with added column2 from df2
    """
    merged_df = (
        df1
        .merge(
            # df2[[team_column, column2]],
            df2,
            how="left",
            left_on=[team_column, column1],
            right_on=[team_column, column2],
        )
    )

    def match_within_team(row):
        if pd.isna(row[column2]):  # Only update if column2 value is null
            team = row[team_column]
            user_name = row[column1]

            roster_names = df2[df2[team_column] == team][column2].dropna().unique().tolist() # list of unique, non-null names from df2[column2] for the same team
            roster_names_lower_map = {normalize_name(n): n for n in roster_names} # match lowercase versions of names, but keep the original versions to return in the end
            result = process.extractOne(normalize_name(user_name), list(roster_names_lower_map.keys()), scorer=fuzz.token_set_ratio, score_cutoff=threshold)
            # Important: two options for a fuzz scorer: token_sort_ratio and token_set_ratio
            # first one sorts tokens and compares two full strings. So, if a fullname has a middle name in one df and doesn't have it in another, it won't be matched
            # second option is better for subset matches as it treats names as sets of tokens and not full strings

            if result:
                return roster_names_lower_map[result[0]]
            return None
        else:
            return row[column2]  # Leave existing column2 value untouched

    merged_df[column2] = merged_df.apply(match_within_team, axis=1)

    return merged_df

In [279]:
participant_info = merge_and_fuzzy_match_names(
                                        df1 = user_info_surveyed,
                                        column1="SurveyName",
                                        df2=df_roster_2022_surveyed,
                                        column2="RosterName",
                                        team_column="Team",
                                        threshold=80
                                    )

In [280]:
participant_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 776 entries, 0 to 775
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Team          776 non-null    object 
 1   UserId        776 non-null    int64  
 2   SurveyName    776 non-null    object 
 3   RosterName    772 non-null    object 
 4   TeamID        707 non-null    float64
 5   Year          707 non-null    float64
 6   Role          707 non-null    object 
 7   Username      707 non-null    object 
 8   RosterUUID    707 non-null    object 
 9   UserUUID      707 non-null    object 
 10  UserIDRoster  707 non-null    float64
dtypes: float64(3), int64(1), object(7)
memory usage: 66.8+ KB


In [281]:
# Drop columns UserUUID, UserIDRoster
participant_info = participant_info.drop(columns=["UserUUID", "UserIDRoster"], errors="ignore")

In [282]:
participant_info[participant_info["RosterName"].isna()]

,Team,UserId,SurveyName,RosterName,TeamID,Year,Role,Username,RosterUUID
0,Cornell,352,Jan Lammerding,None,NaN,NaN,NaN,NaN,NaN
168,Duesseldorf,5891,Tmo Rhiem,None,NaN,NaN,NaN,NaN,NaN
381,NTHU_Taiwan,7651,"Sheng,Bi Chen",None,NaN,NaN,NaN,NaN,NaN
616,VIT_Vellore,9269,Sneha A S,None,NaN,NaN,NaN,NaN,NaN


Unmatched from `surveyed_user_info`:

- First Name: issue in roster API. name completely different but username good. Maybe match based on username if name match not found?
- The rest are a bit different in roster.
- For simplicity, we will add these names manually (But the fuzzy match function can be rewritten that if a match on full name is not found, match on first/last name)

In [283]:
manual_fixes = {
    "Sneha A S": "Sneha Ajay Sunitha",
    "Jan Lammerding": "Nate James Cira",
    "Tmo Rhiem": "Timo Mikel Rhiem",
    "Sheng,Bi Chen": "Sheng Bi, Chen"
}

participant_info.loc[
    participant_info["SurveyName"].isin(manual_fixes),
    "RosterName"
] = participant_info["SurveyName"].map(manual_fixes)

In [284]:
participant_info[participant_info["RosterName"].isna()]

,Team,UserId,SurveyName,RosterName,TeamID,Year,Role,Username,RosterUUID


In [285]:
participant_info = participant_info.sort_values("Team")
participant_info

,Team,UserId,SurveyName,RosterName,TeamID,Year,Role,Username,RosterUUID
56,AFCM-Egypt,5026,Hossam Algamal,Hossam Algamal,4140.0,2022.0,Student Member,hoss0,9d3fed69-3bef-4d73-b4a0-934fbb37d684
55,AFCM-Egypt,5023,Mohamed Sayed Hasouna,Mohamed Sayed Hasouna,4140.0,2022.0,Student Member,mhasouna,b953f098-8f0a-48e8-9b5b-c4ce780491f2
664,ASIJ_Tokyo,9780,Rui Serizawa,Rui Serizawa,4334.0,2022.0,Student Member,rserizawa,fca96926-20eb-4818-9c70-7eee71211c5c
63,ASIJ_Tokyo,5076,Taei Kim,Tei Kim,NaN,NaN,NaN,NaN,NaN
671,ASIJ_Tokyo,9788,Risa Bernier,Risa Bernier,4334.0,2022.0,Student Member,risabernier,c3c52361-edc3-4fa2-96f5-699d18f211ca
...,...,...,...,...,...,...,...,...,...
650,William_and_Mary,9422,Diego Morandi Zerpa,Diego Morandi Zerpa,4174.0,2022.0,Student Member,diegom23,1a6f2245-6cbb-4c8c-ac4b-cab3b02899ac
649,William_and_Mary,9421,Alana Thomas,Alana Thomas,4174.0,2022.0,Student Leader,anthomas01,d89c6359-89bc-4524-8cb8-e779581fd43a
12,William_and_Mary,2240,Avery Bradley,Avery Bradley,4174.0,2022.0,Student Leader,averyb,02b7df22-48aa-4cee-8b9c-6bc9f303a6f8
652,William_and_Mary,9424,Bjorn Shockey,Bjorn Shockey,4174.0,2022.0,Student Member,bbshockey,9d272b1f-23b1-4fbd-b527-ff100bb40dcb


### 1.3 Anonymization

In [286]:
# Function to hash a column in a df with SHA-256

def hash_column(df, column):
    df_copy = df.copy()
    df_copy[column] = df_copy[column].apply(
        lambda x: hashlib.sha256(x.strip().lower().encode('utf-8')).hexdigest()
        if isinstance(x, str) else None
    )
    return df_copy

In [287]:
participant_info_anon = participant_info.copy()
participant_info_anon["RosterNameAnon"] = participant_info_anon["RosterName"]
participant_info_anon = hash_column(participant_info_anon, "RosterNameAnon")
participant_info_anon = participant_info_anon.rename(columns={"UserId": "UserID"})
participant_info_anon = participant_info_anon[['RosterName', 'SurveyName','RosterNameAnon', 'RosterUUID', 'UserID', 'TeamID', 'Team', 'Year']] 

In [288]:
participant_info_anon.to_csv("../data/igem_ties_surveys/2022/participants_matching_table.csv", index=False)

## 2. Merging LLM-extracted task descriptions with manually annotated ones

In [289]:
# Function to load JSON data
def load_attributions_json(filepath):
   with open(filepath, "r", encoding="utf-8") as f:
       data = json.load(f)
   return data

# Function to convert to JSON format since manually annotated teams are now in a TSV files, and turns None into null in JSON
# List of objects where each object has a team name as a key, and the value is a list of members with their names and task descriptions

def convert_df_to_json_format(df):
    if 'Comments' in df.columns:
        df = df.drop(columns=['Comments'])

    json_data = []

    for team, group in df.groupby('Team'):
        records = group[['RosterName', 'RawTextName', 'TasksDescription']].to_dict(orient='records')

        # normalize missing values
        cleaned_records = []
        for record in records:
            cleaned_record = {
                k: (None if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v) else v)
                for k, v in record.items()
            }
            cleaned_records.append(cleaned_record)

        json_data.append({team: cleaned_records})

    return json_data


In [290]:
# Merge text_teams JSON with image_teams, table-teams, 13 evaluation teams, and 7 validation teams
# Some of these are TSV files so convert them to JSON

text_teams_json = load_attributions_json("../data/attributions/2022_attributions/llm_extraction_outputs/text_teams.json")
evaluation_teams_json = load_attributions_json("../data/attributions/2022_attributions/manual_annotation/14_test_teams.json")

validation_teams_df = pd.read_table("../data/attributions/2022_attributions/manual_annotation/7_teams_members_descriptions_from_wikis.tsv")
validation_teams_json = convert_df_to_json_format(validation_teams_df)

image_teams_df = pd.read_table("../data/attributions/2022_attributions/manual_annotation/image_teams.tsv")
image_teams_json = convert_df_to_json_format(image_teams_df)

table_teams_df = pd.read_table("../data/attributions/2022_attributions/manual_annotation/table_teams.tsv")
table_teams_json = convert_df_to_json_format(table_teams_df)

FileNotFoundError: [Errno 2] No such file or directory: '../data/attributions/2022_attributions/manual_annotation/7_teams_members_descriptions_from_wikis.tsv'

In [ ]:
# Remove Teams Aboa and Sogang_Korea from evaluation_teams_json (we will take them from validation_teams_json)

teams_to_remove = {"Aboa", "Sogang_Korea"}

evaluation_teams_json = [
    team_dict
    for team_dict in evaluation_teams_json
    if not any(team_name in teams_to_remove for team_name in team_dict)
]

In [ ]:
# Print all team names in evaluation_teams_json
team_names_in_evaluation = []
for team_dict in evaluation_teams_json:
    team_names_in_evaluation.append(list(team_dict.keys())[0])

team_names_in_evaluation

['Aalto-Helsinki',
 'BostonU_HW',
 'CPU_Nanjing',
 'CSMU_Taiwan',
 'Cambridge',
 'Freiburg',
 'Goettingen',
 'ICT-Mumbai',
 'Montpellier',
 'TU_Braunschweig',
 'Technion-Israel',
 'UPNAvarra_Spain']

In [ ]:
# Text teams have Round1 - remove it so it has the same format as other JSONs

text_teams_json = [
    {
        team_name: (
            # If it's already flattened - keep it as a list of members without nested (so the code can be rerun)
            rounds
            if isinstance(rounds, list)
            # still nested by rounds → flatten
            else [
                member
                for round_entries in rounds.values()
                for member in round_entries
            ]
        )
        for team_name, rounds in team_dict.items()
    }
    for team_dict in text_teams_json
]

In [ ]:
# Merge and save
# BERT and/or LLM classifiers will use this file for final inference for 2022

all_surveyed_teams_task_descriptions = (
    text_teams_json
    + evaluation_teams_json
    + validation_teams_json
    + image_teams_json
    + table_teams_json
)

with open("../data/attributions/2022_attributions/all_surveyed_teams_2022_task_descriptions.json", "w", encoding="utf-8") as f:
    json.dump(
        all_surveyed_teams_task_descriptions,
        f,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
# Check if all teams have been included - there should be 113 surveyed teams

len(all_surveyed_teams_task_descriptions)

112

## 3. Merging final 2022 attributions with survey

### 3.1 Final Attributions Data 

Cleaning BERT and LLM outputs and saving dataframes where one row is one task per person.

In [291]:
# Read BERT/LLM task classes outputs

bert_tasks_df = pd.read_table("../data/attributions/2022_attributions/bert/bert_inference_outputs/bert_classified_tasks_for_surveyed_teams_2022_without_manually_annotated_ones.tsv")

llm_tasks_df = pd.read_table("../data/attributions/2022_attributions/llm_classification_outputs/llm_classified_tasks_for_surveyed_teams_2022_without_manually_annotated_ones.tsv")

# Manually annotated df - take teams and add to bert and llm outputs

manually_annotated_teams = pd.read_table("../data/attributions/2022_attributions/manual_annotation/14_random_teams_tasks.tsv")

# Clean manually annotated df
teams_to_include = ['Aalto-Helsinki','BostonU_HW','CPU_Nanjing','CSMU_Taiwan','Cambridge','Freiburg','Goettingen','ICT-Mumbai','Montpellier','TU_Braunschweig','Technion-Israel','UPNAvarra_Spain']

manually_annotated_teams = (
    manually_annotated_teams
        .drop(columns=["RawTextName", "CommentsLLM", "CommentsBERT"])
        .rename(columns={"RosterName": "FullName"})
        .loc[lambda df: df["Team"].isin(teams_to_include)] # we exclude two teams whose tasks are not complete and are outputs of llm and bert, and we keep teams that we didn't have in bert and llm outputs
)

In [134]:
bert_tasks_df.head()

,Team,FullName,TasksDescription,Tasks
0,Mingdao,Pei-Hong Chen,"As our PI, he passed on his experience to us, ...",[]
1,Mingdao,Hui-Chuan Hsieh,"She is our secondary PI, and taught us how to ...",[]
2,Mingdao,Chen Yu Kao,Responsible for managing and leading part of t...,"['software', 'public engagement', 'writing', '..."
3,Mingdao,Michelle Ching,Responsible for managing and leading part of t...,"['public engagement', 'project administration']"
4,Mingdao,Alex wu,"Debater in GMO scientific debate, translated c...",[]


In [292]:
llm_tasks_df.head()

,Team,FullName,TasksDescription,Tasks,OutputErrors
0,Mingdao,Pei-Hong Chen,"As our PI, he passed on his experience to us, ...","['investigation', 'conceptualization']",NaN
1,Mingdao,Hui-Chuan Hsieh,"She is our secondary PI, and taught us how to ...","['investigation', 'conceptualization']",NaN
2,Mingdao,Chen Yu Kao,Responsible for managing and leading part of t...,"['project administration', 'software', 'public...",NaN
3,Mingdao,Michelle Ching,Responsible for managing and leading part of t...,"['project administration', 'public engagement'...",['Invalid task label: translation']
4,Mingdao,Alex wu,"Debater in GMO scientific debate, translated c...","['public engagement', 'writing']",NaN


In [293]:
df_roster_2022 = pd.read_table("../data/attributions/2022_attributions/team_rosters_2022.tsv")

In [294]:
# Function to clean the BERT/LLM outputs: one row per task instead a list of them, plus cleaning columns and names

def clean_attributions_df(tasks_df):
    df = tasks_df.copy()

    # Drop description column
    df = df.drop(columns=["TasksDescription"], errors="ignore")

    # One task per row
    df["Tasks"] = df["Tasks"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df = df[df["Tasks"].apply(lambda x: isinstance(x, list) and len(x) > 0)].reset_index(drop=True) #keep only tasks that have lists of one or more tasks
    df = df.explode('Tasks')
    df = df.rename(columns={'Tasks': 'Task'})
    df["Team"] = df["Team"].replace("HKU-HongKong", "HKU_HongKong") # fix one team name diff than roster

    return df

In [295]:
bert_tasks_df_cleaned = clean_attributions_df(bert_tasks_df)
llm_tasks_df_cleaned = clean_attributions_df(llm_tasks_df)
manually_annotated_cleaned = clean_attributions_df(manually_annotated_teams)

# Concat manually annotated and bert/llm

bert_merged_df = pd.concat(
    [bert_tasks_df_cleaned, manually_annotated_cleaned],
    ignore_index=True
)

llm_merged_df = pd.concat(
    [llm_tasks_df_cleaned, manually_annotated_cleaned],
    ignore_index=True
)

In [296]:
llm_merged_df

,Team,FullName,Task,OutputErrors
0,Mingdao,Pei-Hong Chen,investigation,NaN
1,Mingdao,Pei-Hong Chen,conceptualization,NaN
2,Mingdao,Hui-Chuan Hsieh,investigation,NaN
3,Mingdao,Hui-Chuan Hsieh,conceptualization,NaN
4,Mingdao,Chen Yu Kao,project administration,NaN
...,...,...,...,...
7490,UPNAvarra_Spain,Unai Gurbindo Gonzalez,writing,NaN
7491,UPNAvarra_Spain,Unai Gurbindo Gonzalez,investigation,NaN
7492,UPNAvarra_Spain,Unai Gurbindo Gonzalez,software,NaN
7493,UPNAvarra_Spain,Unai Gurbindo Gonzalez,visualization,NaN


In [308]:
# Add column for bert probability for each task 
# For those that are manually annotated, just add probability 1


TASK_PROBABILITIES_JSON = "../data/attributions/2022_attributions/bert/bert_inference_outputs/bert_task_probabilities_for_surveyed_teams_2022_without_manually_annotated_ones.json"

with open(TASK_PROBABILITIES_JSON, "r", encoding="utf-8") as f:
    task_probs_json = json.load(f)
# Has structure like: task_probs_json[team][full_name][task] -> probability

def get_task_probability(row, probs_json, manual_teams):
    team = row["Team"]
    name = row["FullName"]
    task = row["Task"]

    # Manually annotated teams - probability = 1
    if team in manual_teams:
        return 1.0
    
    return probs_json[team][name][task]


bert_merged_df["TaskProbability"] = bert_merged_df.apply(
    get_task_probability,
    axis=1,
    probs_json=task_probs_json,
    manual_teams=set(teams_to_include)  
)

In [ ]:
# # Function to prepare attributions dataframe (similar as for years 2023-24-25) with cols from roster "FullName", "Username", "RosterUUID", "TeamID", "Team", "Year", "Role", "Task"

# def merge_outputs_with_roster_df(tasks_df, df_roster_2022):
#     df = tasks_df.copy()

#     roster_cols = [
#         'FullName', 'Username', 'RosterUUID',
#         'TeamID', 'Team', 'Year', 'Role'
#     ]

#     df = df.merge(
#         df_roster_2022[roster_cols],
#         on=['FullName', 'Team'],
#         how='left'
#     )

#     final_columns = [
#         'FullName', 'Username', 'RosterUUID',
#         'TeamID', 'Team', 'Year', 'Role', 'Task'
#     ]

#     return df[final_columns]

The issue with the previous code is that the LLM used for description extraction would sometimes output the wrong roster name - instead of keeping characters like � it would change them into normal letters. So, some names between roster and BERT/LLM need to be normalized and fuzzy matched.

In [332]:
# Normalize names before merging - but issues persist with names having characters like �, so names need to be fuzzy matched
def normalize_name(s):
    if pd.isna(s):
        return s

    try:
        s = s.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return " ".join(s.strip().lower().split())


def merge_outputs_with_roster_df(tasks_df, df_roster_2022, fuzzy_threshold=80):
    df = tasks_df.copy()

    # Normalize names
    df["_name_key"] = df["FullName"].apply(normalize_name)

    roster = df_roster_2022.copy()
    roster["_name_key"] = roster["FullName"].apply(normalize_name)

    roster = roster[
        ["FullName", "Username", "RosterUUID", "TeamID", "Team", "Year", "Role", "_name_key"]
    ].drop_duplicates(subset=["Team", "_name_key"])

    # Exact merge first 
    df = df.merge(
        roster,
        on=["Team", "_name_key"],
        how="left",
        suffixes=("", "_roster")
    )

    # Build team -> roster rows lookup 
    roster_by_team = {
        team: group.set_index("_name_key")
        for team, group in roster.groupby("Team")
    }

    # Fuzzy matching for still-unmatched rows 
    def fill_from_fuzzy(row):
        if pd.notna(row["Username"]):
            return row  # already matched exactly

        team = row["Team"]
        key = row["_name_key"]

        if pd.isna(team) or pd.isna(key):
            return row

        team_roster = roster_by_team.get(team)
        if team_roster is None:
            return row

        match = process.extractOne(
            key,
            team_roster.index,
            scorer=fuzz.WRatio
        )

        if match is None:
            return row

        best_key, score, _ = match
        if score < fuzzy_threshold:
            return row

        roster_row = team_roster.loc[best_key]

        cols = ["FullName", "Username", "RosterUUID", "TeamID", "Year", "Role"]

        row[cols] = roster_row[cols].values

        return row

    df = df.apply(fill_from_fuzzy, axis=1)

    # Cleanup
    df.drop(columns=["_name_key"], inplace=True)

    base_cols = [
    "FullName", "Username", "RosterUUID",
    "TeamID", "Team", "Year", "Role", "Task"
    ]

    final_columns_bert = base_cols + ["TaskProbability"]

    final_columns_llm = base_cols + [ "OutputErrors"]

    try:
        df = df[final_columns_bert] 
    except:
        df = df[final_columns_llm]

    return df

In [333]:
# Save cleaned attributions dataframes to results

bert_final_df = merge_outputs_with_roster_df(
    bert_merged_df,
    df_roster_2022
)

llm_final_df = merge_outputs_with_roster_df(
    llm_merged_df,
    df_roster_2022
)

In [334]:
# Check for unmatched rows - should be 0
unmatched = llm_final_df[llm_final_df['RosterUUID'].isna()]

unmatched[['FullName', 'Team']]

,FullName,Team


In [335]:
bert_final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4040 entries, 0 to 4039
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   FullName         4040 non-null   object 
 1   Username         4040 non-null   object 
 2   RosterUUID       4040 non-null   object 
 3   TeamID           4040 non-null   float64
 4   Team             4040 non-null   object 
 5   Year             4040 non-null   float64
 6   Role             4040 non-null   object 
 7   Task             4040 non-null   object 
 8   TaskProbability  4040 non-null   float64
dtypes: float64(3), object(6)
memory usage: 284.2+ KB


In [336]:
llm_final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7495 entries, 0 to 7494
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   FullName      7495 non-null   object 
 1   Username      7495 non-null   object 
 2   RosterUUID    7495 non-null   object 
 3   TeamID        7495 non-null   float64
 4   Team          7495 non-null   object 
 5   Year          7495 non-null   float64
 6   Role          7495 non-null   object 
 7   Task          7495 non-null   object 
 8   OutputErrors  508 non-null    object 
dtypes: float64(2), object(7)
memory usage: 527.1+ KB


In [337]:
bert_final_df['Task'] = bert_final_df['Task'].astype(str).str.capitalize()
llm_final_df['Task'] = llm_final_df['Task'].astype(str).str.capitalize()

bert_final_df.to_csv("../results/attributions/attributions_for_surveyed_teams_bert_2022.tsv", sep="\t", index=False)
llm_final_df.to_csv("../results/attributions/attributions_for_surveyed_teams_llm_2022.tsv", sep="\t", index=False)

### 3.2 Attributions and Survey Combined

Combining the previous final tasks dataframes with the survey.

In [338]:
survey_tasks_df = pd.read_csv("../data/igem_ties_surveys/2022/survey_tasks.csv")
participant_info_anon = pd.read_csv("../data/igem_ties_surveys/2022/participants_matching_table.csv")

In [339]:
# Add "RosterName" from participant_info_anon to survey by merging on "user_id" in survey, or "UserID" (in partcipant info)
# Check if everything merged - no empty RosterName

survey_tasks_df = (
    survey_tasks_df
    .merge(
        participant_info_anon[["RosterName", "UserID"]],
        how="left",
        left_on="user_id",
        right_on="UserID",
    )
    .drop(columns=["UserID", "user_id"])
)

survey_tasks_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11610 entries, 0 to 11609
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   team        11610 non-null  object
 1   Task        11610 non-null  object
 2   Certainty   11489 non-null  object
 3   Experience  11007 non-null  object
 4   Preference  10779 non-null  object
 5   RosterName  11610 non-null  object
dtypes: object(6)
memory usage: 544.3+ KB


In [340]:
# Merge survey_tasks_df and BERT/LLM on "RosterName" in survey, or "FullName" in BERT/LLM

# Keep Lab Maitenance as null
# Keep columns: Team	TeamMember	Role	Task	Certainty	Experience	Preference	TaskPerformed

def combine_survey_tasks_with_attributions(survey_df, attributions_df,
                                           s_task_col, a_task_col,
                                           s_participant_col, a_participant_col):
    """
    Combine survey tasks with attributions, adding 'TaskPerformed' and 'Role' columns gathered from attributions.
    
    Parameters:
    - survey_df (DataFrame): dataframe with survey task responses
    - attributions_df (DataFrame): dataframe with team attributions (bert or llm final outputs in this case)
    - s_task_col (str): column name in survey_df for task names ('Task')
    - a_task_col (str): column name in attributions_df for task names ('Task')
    - s_participant_col (str): column name in survey_df for participant hash ('RosterName')
    - a_participant_col (str): column name in attributions_df for participant hash ('FullName')

    Returns:
    - df: a copy of survey_df with an additional 'TaskPerformed' column:
        - True: participant-task pair exists in attributions
        - False: participant exists, but task not TaskPerformed
        - NA: participant not found in attributions and null for task named "Lab Maitenance"
    and 'Role' column - the participant's role from attributions_df, matched by participant name.
    """
    df = survey_df.copy()
    attributions_copy = attributions_df.copy()

    # Create merge key for exact participant-task match
    df['_merge_key'] = df[s_participant_col] + '||' + df[s_task_col]
    attributions_copy['_merge_key'] = attributions_copy[a_participant_col] + '||' + attributions_copy[a_task_col]

    # Assign TaskPerformed based on composite key
    df['TaskPerformed'] = df['_merge_key'].isin(attributions_copy['_merge_key']).astype('Int64')

    # Set TaskPerformed = pd.NA where participant is not found in attributions
    unmatched_participants = ~df[s_participant_col].isin(attributions_copy[a_participant_col])
    df.loc[unmatched_participants, 'TaskPerformed'] = pd.NA

    # Lab Maintenance performance as pd.NA 
    df.loc[df[s_task_col] == "Lab Maintenance", "TaskPerformed"] = pd.NA

    # Add Role by merging on participant only (not task), avoiding column collision
    roles = attributions_copy[[a_participant_col, 'Role']].drop_duplicates(subset=[a_participant_col])
    roles = roles.rename(columns={'Role': '_Role'})  # temporary name to avoid conflict
    df = df.merge(roles, left_on=s_participant_col, right_on=a_participant_col, how='left')
    df.rename(columns={'_Role': 'Role'}, inplace=True)

    # Cleanup
    df.drop(columns=['_merge_key', a_participant_col], inplace=True)

    return df

In [341]:
combined_bert_df = combine_survey_tasks_with_attributions(
    survey_df=survey_tasks_df,
    attributions_df=bert_final_df,
    s_task_col='Task',
    a_task_col='Task',
    s_participant_col='RosterName',
    a_participant_col='FullName'
)

combined_llm_df = combine_survey_tasks_with_attributions(
    survey_df=survey_tasks_df,
    attributions_df=llm_final_df,
    s_task_col='Task',
    a_task_col='Task',
    s_participant_col='RosterName',
    a_participant_col='FullName'
)

In [343]:
combined_bert_df.rename(columns={'RosterName': 'TeamMember', 'team': 'Team'}, inplace=True)
combined_llm_df.rename(columns={'RosterName': 'TeamMember', 'team': 'Team'}, inplace=True)

new_column_order = [
    "Team",
    "TeamMember",  
    "Role", 
    "Task", 
    "Certainty",
    "Experience",
    "Preference",
    "TaskPerformed"
]

combined_bert_df = combined_bert_df[new_column_order]
combined_llm_df = combined_llm_df[new_column_order]

In [219]:
# Total number of rows
total_rows = len(combined_bert_df)

# Number of rows where TaskPerformed == 0
task_not_performed = (combined_bert_df['TaskPerformed'] == 0).sum()

# Number of rows where TaskPerformed == 1
task_performed = (combined_bert_df['TaskPerformed'] == 1).sum()

# Number of rows where TaskPerformed is null - not counting in Lab Maintenance
mask = (
    combined_bert_df["TaskPerformed"].isna() &
    (combined_bert_df["Task"] != "Lab Maintenance")
)
task_performed_null = mask.sum()


print(f"Total rows in combined dataframe: {total_rows}")
print(f"TaskPerformed = 0: {task_not_performed}")
print(f"TaskPerformed = 1: {task_performed}")
print(f"TaskPerformed = NA (not counting in Lab Maintenance): {task_performed_null}")

Total rows in combined dataframe: 11610
TaskPerformed = 0: 7631
TaskPerformed = 1: 1250
TaskPerformed = NA (not counting in Lab Maintenance): 1953


The null values for `TaskPerformed`, not counting in task "Lab Maintenance", signify that BERT/LLM did not output any task for that person.

In [344]:
# Hah name

combined_bert_df_anon = hash_column(combined_bert_df, "TeamMember")
combined_llm_df_anon = hash_column(combined_llm_df, "TeamMember")

In [262]:
# Check if everything matches between participants_matching_table and combined survey attribution

anon_matching_check = pd.read_csv("../data/igem_ties_surveys/2022/participants_matching_table.csv")
anon_matching_check = anon_matching_check.merge(combined_bert_df_anon, left_on=["RosterNameAnon"], right_on=['TeamMember'], how='right')

unmatched = anon_matching_check[
    anon_matching_check["RosterNameAnon"].isna()
]
unmatched

,RosterName,SurveyName,RosterNameAnon,RosterUUID,UserID,TeamID,Team_x,Year,Team_y,TeamMember,Role,Task,Certainty,Experience,Preference,TaskPerformed


In [345]:
combined_llm_df_anon.to_csv("../results/survey_tasks_and_attributions/2022_survey_tasks_and_attributions_combined_llm.tsv", sep="\t", index=False)
combined_bert_df_anon.to_csv("../results/survey_tasks_and_attributions/2022_survey_tasks_and_attributions_combined_bert.tsv", sep="\t", index=False)